<a href="https://colab.research.google.com/github/I-AM-PRASHANT-VERMA/Almabetter-voyage-analytics-mlops/blob/main/Gender_classification_ML_model/Gender_classification_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 📌 Project Summary – Gender Classification Model 👤🤖

This project focuses on building a simple yet effective **gender classification system** using user data from the travel dataset. The main idea was to predict gender based on the user’s first name, which was extracted and cleaned from the original dataset. After preprocessing, duplicate removal, and filtering, a well-balanced dataset was prepared to ensure fair model training. Multiple machine learning models were applied, including Logistic Regression, Naive Bayes, and SVM, to find the best-performing approach. Among them, the **tuned Linear SVM model** performed the best with around **90% accuracy**, making it a reliable choice for prediction.

The model was evaluated using standard metrics like accuracy, precision, recall, and F1-score, along with confusion matrix analysis to understand prediction quality. The results showed strong and consistent performance across both classes, indicating that the model is stable and not biased. This classification system can be useful in real-world applications such as user segmentation, personalization, and targeted recommendations in travel platforms. Overall, this project demonstrates how basic text data can be transformed into meaningful insights using machine learning, and it also contributes as an important component in the complete MLOps-based travel analytics system 🚀


In [ ]:
# ==============================
# 1. Importing Important Libraries
# ==============================

import pandas as pd  # Used for reading and handling dataset tables.
import numpy as np  # Used for numerical operations when needed.
import joblib  # Used for saving and loading the final trained model.
from google.colab import drive  # Used for connecting Google Drive in Colab.
from sklearn.model_selection import train_test_split  # Used for splitting data into train and test sets.
from sklearn.pipeline import Pipeline  # Used for combining TF-IDF and model into one workflow.
from sklearn.feature_extraction.text import TfidfVectorizer  # Used for converting names into text-based numeric features.
from sklearn.linear_model import LogisticRegression  # Used as one lightweight classification model.
from sklearn.naive_bayes import MultinomialNB  # Used as another simple classification model.
from sklearn.svm import LinearSVC  # Used as a strong lightweight text classification model.
from sklearn.metrics import accuracy_score  # Used for checking overall prediction accuracy.
from sklearn.metrics import precision_score  # Used for checking how correct positive predictions are.
from sklearn.metrics import recall_score  # Used for checking how well actual classes are detected.
from sklearn.metrics import f1_score  # Used for checking balanced model performance.
from sklearn.metrics import classification_report  # Used for detailed performance report.
from sklearn.metrics import confusion_matrix  # Used for checking correct and wrong predictions by class.

print("Libraries imported successfully.")  # Confirms that all required libraries are available.


Libraries imported successfully.


In [ ]:
# ==============================
# 2. Mounting Google Drive
# ==============================

drive.mount('/content/drive')  # Connects Google Drive with the Colab notebook.

print("Google Drive mounted successfully.")  # Confirms that Drive is ready to use.


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully.


In [ ]:
# ==============================
# 3. Loading The Dataset Files
# ==============================

users_path = '/content/drive/MyDrive/Colab Notebooks/# specialization projects/1. Voyage Analytics: Integrating MLOps in Travel/dataset flight hotel users/users.csv'  # Stores users dataset path.
flights_path = '/content/drive/MyDrive/Colab Notebooks/# specialization projects/1. Voyage Analytics: Integrating MLOps in Travel/dataset flight hotel users/flights.csv'  # Stores flights dataset path.
hotels_path = '/content/drive/MyDrive/Colab Notebooks/# specialization projects/1. Voyage Analytics: Integrating MLOps in Travel/dataset flight hotel users/hotels.csv'  # Stores hotels dataset path.

users = pd.read_csv(users_path)  # Loads the users dataset.
flights = pd.read_csv(flights_path)  # Loads the flights dataset.
hotels = pd.read_csv(hotels_path)  # Loads the hotels dataset.

print("Users dataset shape:", users.shape)  # Shows rows and columns in users data.
print("Flights dataset shape:", flights.shape)  # Shows rows and columns in flights data.
print("Hotels dataset shape:", hotels.shape)  # Shows rows and columns in hotels data.

print("\nUsers dataset preview:")  # Prints a heading before previewing users data.
display(users.head())  # Displays the first five rows of users data.


Users dataset shape: (1340, 5)
Flights dataset shape: (271888, 10)
Hotels dataset shape: (40552, 8)

Users dataset preview:


,code,company,name,gender,age
0,0,4You,Roy Braun,male,21
1,1,4You,Joseph Holsten,male,37
2,2,4You,Wilma Mcinnis,female,48
3,3,4You,Paula Daniel,female,23
4,4,4You,Patricia Carson,female,44


✅ The dataset loading step was completed successfully. I loaded three files: `users`, `flights`, and `hotels`. The `users` dataset contains 1340 records and 5 columns, while the `flights` and `hotels` datasets are much larger, with 271888 and 40552 records respectively.

For this gender classification task, the `users` dataset is the main dataset because it includes the required `name` and `gender` columns. The preview also shows that the data has useful columns like `code`, `company`, `name`, `gender`, and `age`. Since the model is going to predict gender from names, I will mainly work with the `name` and `gender` columns in the next steps.

In [ ]:
# ==============================
# 4. Selecting And Cleaning Required Data
# ==============================

df = users[['name', 'gender']].copy()  # Keeps only the required columns for gender classification.

df = df.dropna(subset=['name', 'gender'])  # Removes rows where name or gender is missing.

df['name'] = df['name'].astype(str).str.strip()  # Converts names into string format and removes extra spaces.

df['gender'] = df['gender'].astype(str).str.lower().str.strip()  # Converts gender values to lowercase and removes extra spaces.

df = df[df['gender'].isin(['male', 'female'])].copy()  # Keeps only male and female classes.

df = df[df['name'].str.len() > 1].copy()  # Removes names that are too short to be useful.

df = df.drop_duplicates(subset=['name', 'gender']).copy()  # Removes duplicate name-gender records.

df['first_name'] = df['name'].str.split().str[0]  # Extracts first name because it is more useful for gender prediction.

df = df.reset_index(drop=True)  # Resets row numbering after cleaning.

print("Cleaned dataset shape:", df.shape)  # Shows the final dataset size after cleaning.

print("\nGender distribution:")  # Prints a heading for class balance.
print(df['gender'].value_counts())  # Shows the number of male and female records.

print("\nCleaned dataset preview:")  # Prints a heading before previewing cleaned data.
display(df.head())  # Displays the first five cleaned rows.


Cleaned dataset shape: (900, 3)

Gender distribution:
gender
male      452
female    448
Name: count, dtype: int64

Cleaned dataset preview:


,name,gender,first_name
0,Roy Braun,male,Roy
1,Joseph Holsten,male,Joseph
2,Wilma Mcinnis,female,Wilma
3,Paula Daniel,female,Paula
4,Patricia Carson,female,Patricia


✅ In this step, I cleaned the users dataset and kept only the columns needed for the gender classification task. After cleaning, the dataset contains **900 records and 3 columns**: **`name`**, **`gender`**, and the new **`first_name`** column.

The important change here is that I created **`first_name`** from the full name. This was done because gender prediction usually depends more on the first name than the surname. For example, in **“Patricia Carson”**, the name **“Patricia”** gives more useful gender information than **“Carson”**.

The class distribution is also almost balanced, with **452 male** records and **448 female** records. This is good for training because the model gets nearly equal examples from both classes. The preview confirms that the new **`first_name`** column was created correctly, like **Roy**, **Joseph**, **Wilma**, **Paula**, and **Patricia**.

In [ ]:
# ==============================
# 5. Splitting Data Into Training And Testing
# ==============================

X = df['first_name']  # Uses first name as the input feature.

y = df['gender']  # Uses gender as the target label.

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)  # Splits data into 80% training and 20% testing with balanced classes.

print("Training data size:", X_train.shape[0])  # Shows the number of training records.

print("Testing data size:", X_test.shape[0])  # Shows the number of testing records.

print("\nTraining label distribution:")  # Prints a heading for training class balance.
print(y_train.value_counts())  # Shows male and female counts in training data.

print("\nTesting label distribution:")  # Prints a heading for testing class balance.
print(y_test.value_counts())  # Shows male and female counts in testing data.


Training data size: 720
Testing data size: 180

Training label distribution:
gender
male      362
female    358
Name: count, dtype: int64

Testing label distribution:
gender
female    90
male      90
Name: count, dtype: int64


✅ In this step, I used the **`first_name`** column as the input feature and **`gender`** as the target column. Then I split the data into training and testing sets using an **80/20 split**.

The output shows that **720 records** were used for training and **180 records** were kept for testing. This gives the model enough data to learn from, while still keeping a separate part of the dataset to check how well it performs on unseen data.

The split is also balanced. In the training data, there are **362 male** and **358 female** records, and in the testing data, there are exactly **90 female** and **90 male** records. This is useful because the model will not be tested unfairly on one class more than the other. The **`stratify=y`** part helped maintain this balance.

In [ ]:
# ==============================
# 6. Creating And Training Multiple Models
# ==============================

models = {  # Stores different lightweight models for comparison.
    "Logistic Regression": LogisticRegression(max_iter=1000),  # Uses Logistic Regression for text classification.
    "Naive Bayes": MultinomialNB(),  # Uses Naive Bayes as a simple baseline model.
    "Linear SVM": LinearSVC(),  # Uses Linear SVM as a strong lightweight classifier.
    "Linear SVM Tuned": LinearSVC(C=0.5)  # Uses a slightly tuned SVM version for comparison.
}  # Ends the model dictionary.

results = []  # Stores evaluation scores of each model.

trained_models = {}  # Stores trained model pipelines.

for model_name, model in models.items():  # Runs training for each model one by one.
    pipeline = Pipeline([  # Creates a complete training pipeline.
        ('tfidf', TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 6))),  # Converts first names into character n-gram TF-IDF features.
        ('model', model)  # Adds the classifier after TF-IDF.
    ])  # Ends the pipeline setup.

    pipeline.fit(X_train, y_train)  # Trains the pipeline on training data.

    y_pred = pipeline.predict(X_test)  # Predicts gender on test data.

    accuracy = accuracy_score(y_test, y_pred)  # Calculates accuracy score.

    precision = precision_score(y_test, y_pred, average='weighted')  # Calculates weighted precision score.

    recall = recall_score(y_test, y_pred, average='weighted')  # Calculates weighted recall score.

    f1 = f1_score(y_test, y_pred, average='weighted')  # Calculates weighted F1-score.

    results.append([model_name, accuracy, precision, recall, f1])  # Saves the model scores.

    trained_models[model_name] = pipeline  # Saves the trained pipeline for later selection.

    print("\nModel trained:", model_name)  # Shows which model finished training.
    print("Accuracy:", round(accuracy, 4))  # Shows rounded accuracy.
    print("F1 Score:", round(f1, 4))  # Shows rounded F1-score.



Model trained: Logistic Regression
Accuracy: 0.8722
F1 Score: 0.8721

Model trained: Naive Bayes
Accuracy: 0.8722
F1 Score: 0.8721

Model trained: Linear SVM
Accuracy: 0.9
F1 Score: 0.8998

Model trained: Linear SVM Tuned
Accuracy: 0.9
F1 Score: 0.8998


✅ In this step, I trained **four lightweight machine learning models** and compared how they performed on the test data. The models were **Logistic Regression**, **Naive Bayes**, **Linear SVM**, and a slightly tuned version called **Linear SVM Tuned**.

For feature extraction, I used **TF-IDF with character n-grams** on the **first names**. This means the model learns small character patterns from names, like starting letters, ending letters, and common letter combinations. This is useful for gender classification because names often have repeated patterns.

The output shows a clear improvement compared to the earlier full-name version. **Logistic Regression** and **Naive Bayes** both reached about **87.22% accuracy**, while **Linear SVM** and **Linear SVM Tuned** reached **90% accuracy** with an F1-score around **0.8998**.

This is a strong result for a lightweight machine learning model. There is no problem in the output. The main point is that using **first names** helped the model perform better, and **Linear SVM** is currently the strongest model from this set.

In [ ]:
# ==============================
# 7. Comparing All Model Results
# ==============================

results_df = pd.DataFrame(results, columns=['Model', 'Accuracy', 'Precision', 'Recall', 'F1 Score'])  # Converts all model scores into a table.

results_df = results_df.sort_values(by='F1 Score', ascending=False).reset_index(drop=True)  # Sorts models from best to worst using F1-score.

print("Model comparison table:")  # Prints a heading for the comparison table.

display(results_df)  # Displays the model comparison table.

best_model_name = results_df.loc[0, 'Model']  # Selects the model with the highest F1-score.

best_model = trained_models[best_model_name]  # Gets the trained pipeline of the best model.

print("Best model selected:", best_model_name)  # Displays the selected best model.


Model comparison table:


,Model,Accuracy,Precision,Recall,F1 Score
0,Linear SVM Tuned,0.900000,0.903186,0.900000,0.899802
1,Linear SVM,0.900000,0.903186,0.900000,0.899802
2,Naive Bayes,0.872222,0.873375,0.872222,0.872124
3,Logistic Regression,0.872222,0.873375,0.872222,0.872124


Best model selected: Linear SVM Tuned


✅ In this step, I compared all the trained models in one table using **Accuracy**, **Precision**, **Recall**, and **F1 Score**. I mainly used **F1 Score** for selecting the best model because it gives a balanced view of the model’s performance.

From the output, **Linear SVM Tuned** and **Linear SVM** both gave the same performance, with **90% accuracy** and an **F1 Score of 0.899802**. Since both models performed equally, there is not a big difference between choosing either one.

The code selected **Linear SVM Tuned** as the best model because it appeared at the top after sorting the results. This is not a problem. It just means that the tuned version with **C = 0.5** gave the same result as the normal Linear SVM, so we can continue with **Linear SVM Tuned** as the final model.

Compared to Logistic Regression and Naive Bayes, the SVM models performed better, so selecting **Linear SVM Tuned** for the final `.joblib` file is a reasonable choice.

In [ ]:
# ==============================
# 8. Detailed Evaluation Of The Best Model
# ==============================

best_predictions = best_model.predict(X_test)  # Predicts the test data using the selected best model.

print("Best Model:", best_model_name)  # Displays the best model name.

print("\nClassification Report:")  # Prints a heading for detailed metrics.
print(classification_report(y_test, best_predictions))  # Shows precision, recall, F1-score, and support.

print("\nConfusion Matrix:")  # Prints a heading for confusion matrix.
print(confusion_matrix(y_test, best_predictions))  # Shows correct and incorrect predictions for both classes.


Best Model: Linear SVM Tuned

Classification Report:
              precision    recall  f1-score   support

      female       0.94      0.86      0.90        90
        male       0.87      0.94      0.90        90

    accuracy                           0.90       180
   macro avg       0.90      0.90      0.90       180
weighted avg       0.90      0.90      0.90       180


Confusion Matrix:
[[77 13]
 [ 5 85]]


✅ In this step, I checked the detailed performance of the selected best model, **Linear SVM Tuned**.

The model achieved **90% accuracy** on the test data, which is a good improvement compared to the earlier full-name version. The test set had **90 female** and **90 male** records, so the evaluation is balanced.

From the report, the model performed well for both classes. For **female**, the precision is **0.94**, which means when the model predicts female, it is usually correct. For **male**, the recall is **0.94**, which means it correctly identifies most of the actual male names.

The confusion matrix gives a clearer picture:

- **77 female names** were correctly predicted as female.
- **13 female names** were wrongly predicted as male.
- **85 male names** were correctly predicted as male.
- **5 male names** were wrongly predicted as female.

So the model is working well overall, but it still makes more mistakes by classifying some **female names as male**. This is not a coding issue; it just shows the model still has some limitation. Since the overall accuracy and F1-score are both around **90%**, this model is a strong final choice for the project.

In [ ]:
# ==============================
# 9. Testing The Best Model With Sample Names
# ==============================

sample_names = ["Roy", "Patricia", "David", "Amit", "Anjali", "Priya", "John"]  # Creates sample first names for testing.

sample_predictions = best_model.predict(sample_names)  # Predicts gender for the sample names.

sample_output = pd.DataFrame({  # Creates a table for sample prediction results.
    "Name": sample_names,  # Stores the sample names.
    "Predicted Gender": sample_predictions  # Stores the predicted gender values.
})  # Ends the sample result table.

print("Sample prediction results:")  # Prints a heading for sample predictions.

display(sample_output)  # Displays the sample prediction table.


Sample prediction results:


,Name,Predicted Gender
0,Roy,male
1,Patricia,female
2,David,male
3,Amit,male
4,Anjali,female
5,Priya,female
6,John,male


✅ In this step, I tested the final selected model with some sample first names to check how it predicts on normal inputs.

The sample results look good. The model predicted **Roy**, **David**, **Amit**, and **John** as **male**, while **Patricia**, **Anjali**, and **Priya** were predicted as **female**. These predictions match the expected gender for these common sample names.

This is also an improvement from the earlier version, where **Amit** was wrongly predicted as female when the model was trained on full names. After switching to **first names**, the prediction for Amit became correct. So this sample test supports the decision to use **first_name** instead of full `name` for training.

In [ ]:
# ==============================
# 10. Saving The Final Best Model As Joblib File
# ==============================

final_model_path = "/content/drive/MyDrive/gender_classifier_best_model.joblib"  # Sets the Google Drive location for the final model.

joblib.dump(best_model, final_model_path)  # Saves the complete TF-IDF and classifier pipeline as one joblib file.

print("Final model saved successfully at:", final_model_path)  # Displays the saved model path.


Final model saved successfully at: /content/drive/MyDrive/gender_classifier_best_model.joblib


In [ ]:
# ==============================
# 11. Loading The Saved Joblib Model And Testing Again
# ==============================

loaded_model = joblib.load("/content/drive/MyDrive/gender_classifier_best_model.joblib")  # Loads the saved joblib model from Google Drive.

test_name = ["Amit"]  # Creates one sample first name for final checking.

loaded_prediction = loaded_model.predict(test_name)  # Predicts gender using the loaded model.

print("Test name:", test_name[0])  # Displays the test name.

print("Predicted gender from loaded model:", loaded_prediction[0])  # Displays the prediction from the loaded model.


Test name: Amit
Predicted gender from loaded model: male


### ✅ Final Conclusion of Gender Classification Model 👤⚙️

This project successfully built a **gender classification model** using user name data, where the first name was used as the key feature to predict gender. After proper data cleaning, preprocessing, and removing inconsistencies, a balanced dataset of 900 records was created, ensuring fair model learning. Multiple machine learning models were trained and compared, where **Linear SVM (Tuned)** achieved the best performance with around **90% accuracy and F1-score**, showing strong predictive capability. The confusion matrix and classification report confirmed that the model performs well for both male and female classes with minimal bias. Additionally, testing on real-world sample names demonstrated that the model is practical and reliable for prediction tasks. Overall, this model proves that even simple text-based features like names can be effectively used for classification, and it fits well into the larger travel analytics system for personalization and user profiling, supporting the broader MLOps pipeline goals of the project  🚀
